<a href="https://colab.research.google.com/github/iraisheredia/IA-APLICACA-A-MODELOS/blob/main/Copia_de_Hands_On_Prompt_Engineering_y_Sistemas_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq --quiet
from groq import Groq
from google.colab import userdata
client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.1 MB/s eta 0:00:00
Cliente de Groq inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [ ]:
# Prompt de clasificación en modo zero-shot
prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."

response_zero = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_zero_shot}]
)

print("Zero-shot:", response_zero.choices[0].message.content)

Zero-shot: Mixto


In [ ]:
# Prompt de clasificación en modo few-shot
prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo

Reseña: "El envío llegó tarde pero el producto es excelente."
Sentimiento:"""

response_few = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_few_shot}]
)

print("Few-shot:", response_few.choices[0].message.content)
# El formato few-shot suele acotar mejor la salida a una sola palabra de la categoría esperada

Few-shot: Sentimiento: Mixto


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [ ]:
# Razonamiento paso a paso (chain-of-thought)
problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.choices[0].message.content)
# problema_directo = problema.split(".")[-2]  # variante: pedir solo el número, sin razonamiento, y comparar qué tan seguido se equivoca


**Paso a paso**

1. **Definir las velocidades**  
   - Primer tren: \(v_1 = 80 \text{ km/h}\)  
   - Segundo tren: \(v_2 = 120 \text{ km/h}\)

2. **Establecer el tiempo de partida**  
   - El primer tren sale a \(t = 0\).  
   - El segundo tren sale 2 h después, es decir, a \(t = 2 \text{ h}\).

3. **Elegir una variable de tiempo**  
   Sea \(t\) el tiempo transcurrido **después** de que el segundo tren salga (es decir, tiempo que el segundo tren ha estado en movimiento).  
   Entonces:
   - El segundo tren ha viajado durante \(t\) horas.
   - El primer tren ha viajado durante \(2 + t\) horas (porque ya llevaba 2 h cuando el segundo partió).

4. **Expresar las distancias recorridas**  
   \[
   \begin{aligned}
   \text{Distancia del primer tren} &= v_1 \times (2 + t) = 80(2 + t) \text{ km} \\
   \text{Distancia del segundo tren} &= v_2 \times t = 120t \text{ km}
   \end{aligned}
   \]

5. **Igualar las distancias** (condición de encuentro)  
   \[
   80(2 + t) = 120t
   \]

6. **Resolv

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [ ]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina
prompt_desconocido = (
    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de "
    "2026? Respuesta muy breve y corta."
)

response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.choices[0].message.content)
# Por más segura que suene la respuesta, el modelo no tiene forma de saber esto: es una alucinación

El equipo **Nexus** ganó la final con su proyecto **“SmartLog”**.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [ ]:
# Instalar sentence-transformers
!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

In [ ]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "
    "empaque original.",
    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "
    "envío de regreso.",
    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "
    "cambio de talla."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


In [ ]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Puedo devolver algo que compré en oferta?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


In [ ]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG
prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_rag.choices[0].message.content)
# Compara esta respuesta contra lo que el modelo diría sin el fragmento: sin RAG probablemente improvisaría una política genérica

No, los productos en oferta no son elegibles para devolución, solo para cambio de talla.


# **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

## **INSTRUCCIONES:**

**1. Define tu base de conocimiento y genera embeddings:**

* Lee la API key previamente configurada desde **Colab Secrets** e instala/importa las librerías necesarias.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings con `sentence-transformers`.

In [9]:
# Leer API key, instalar e importar librerías
!pip install groq --quiet
from groq import Groq
from google.colab import userdata
client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.6 MB/s eta 0:00:00
Cliente de Groq inicializado correctamente.


In [6]:
from sentence_transformers import SentenceTransformer

# Definir la lista documentos y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "El cliente podrá solicitar la devolución o cambio del producto si presenta defecto de fabrica, siempre que lo reporte dentro de los 15 días posteriores a la compra y presente el ticket",
    "El huesped podrá calcelar su reservación sin penalización hasta 48 horas antes de la fecha de llegada.",
    "Cuando un paquete no pueda ser entregado debido a direccion incorrecta,destinatario ausente o rechazo del envió la empresa realizara hasta dos intentos de entrega ",

]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [7]:
import numpy as np

# Definir la función buscar_fragmento

def buscar_fragmento(pregunta):
  embedding_pregunta = modelo_embeddings.encode([pregunta])
  similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
  indice_mas_similar = np.argmax(similitudes)
  return documentos[indice_mas_similar]

pregunta = "¿El cliente puede solicitar la devolucion o cambio del producto?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: El cliente podrá solicitar la devolución o cambio del producto si presenta defecto de fabrica, siempre que lo reporte dentro de los 15 días posteriores a la compra y presente el ticket


**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [12]:
preguntas_sin_rag = [
    "¿El cliente puede solicitar la devolucion o cambio del producto?",
    "¿Cuándo puede un huésped cancelar su reservación sin penalización?",
    "¿Qué sucede si un paquete no puede ser entregado?"
]

for i, p in enumerate(preguntas_sin_rag):
    print(f"\n--- Pregunta {i+1} sin RAG ---")
    prompt_sin_rag = f"Responde la siguiente pregunta: {p}"

    response_sin_rag = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt_sin_rag}]
    )

    respuesta_sin_rag = response_sin_rag.choices[0].message.content
    print("Respuesta sin RAG:", respuesta_sin_rag)



--- Pregunta 1 sin RAG ---
Respuesta sin RAG: Sí, el cliente puede solicitar la devolución o el cambio de un producto, siempre y cuando cumpla con los requisitos establecidos por la política de devoluciones y cambios de la tienda. A continuación, te detallo los puntos clave que suelen aplicarse:

| Condición | Detalle típico |
|-----------|----------------|
| **Plazo** | La solicitud debe hacerse dentro de un período determinado (por ejemplo, 30 días) después de la recepción del producto. |
| **Estado del producto** | El artículo debe estar en su condición original, sin uso, con todas las etiquetas, embalaje original y accesorios incluidos. |
| **Motivo** | Se aceptan devoluciones por defectos de fabricación, daños durante el envío, errores de envío (producto equivocado o mal especificado) o insatisfacción por razones de gusto. |
| **Proceso** | 1. Contactar al servicio al cliente (correo, chat, teléfono). 2. Obtener una etiqueta de devolución (si corresponde). 3. Empacar el producto 

**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [14]:
preguntas_rag = [
    "¿El cliente puede solicitar la devolucion o cambio del producto?",
    "¿Cuándo puede un huésped cancelar su reservación sin penalización?",
    "¿Qué sucede si un paquete no puede ser entregado?"
]

for i, p in enumerate(preguntas_rag):
    print(f"\n--- Pregunta {i+1} con RAG ---")
    fragmento_actual = buscar_fragmento(p)
    prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política del hotel. Si la política no cubre la pregunta, dilo claramente.\n\nPolítica: {fragmento_actual}\n\nPregunta: {p}\n\nRespuesta muy breve y corta."""

    response_rag = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt_rag}]
    )

    respuesta_rag = response_rag.choices[0].message.content
    print("Respuesta con RAG:", respuesta_rag)



--- Pregunta 1 con RAG ---
Respuesta con RAG: Sí, si el producto tiene defecto de fábrica, el cliente puede solicitar devolución o cambio dentro de los 15 días posteriores a la compra y presentando el ticket.

--- Pregunta 2 con RAG ---
Respuesta con RAG: Hasta 48 horas antes de la fecha de llegada.

--- Pregunta 3 con RAG ---
Respuesta con RAG: Se harán hasta dos intentos de entrega.


In [15]:
import pandas as pd

# Re-ejecutar la lógica para recopilar todas las respuestas

preguntas = [
    "¿El cliente puede solicitar la devolucion o cambio del producto?",
    "¿Cuándo puede un huésped cancelar su reservación sin penalización?",
    "¿Qué sucede si un paquete no puede ser entregado?"
]

respuestas_sin_rag_lista = []
respuestas_rag_lista = []

for p in preguntas:
    # Generar respuesta sin RAG
    prompt_sin_rag = f"Responde la siguiente pregunta: {p}"
    response_sin_rag = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt_sin_rag}]
    )
    respuestas_sin_rag_lista.append(response_sin_rag.choices[0].message.content)

    # Generar respuesta con RAG
    fragmento_actual = buscar_fragmento(p)
    prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política del hotel. Si la política no cubre la pregunta, dilo claramente.\n\nPolítica: {fragmento_actual}\n\nPregunta: {p}\n\nRespuesta muy breve y corta."""
    response_rag = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt_rag}]
    )
    respuestas_rag_lista.append(response_rag.choices[0].message.content)

# Crear DataFrame para comparar
df_comparacion = pd.DataFrame({
    'Pregunta': preguntas,
    'Respuesta sin RAG': respuestas_sin_rag_lista,
    'Respuesta con RAG': respuestas_rag_lista
})

display(df_comparacion)


,Pregunta,Respuesta sin RAG,Respuesta con RAG
0,¿El cliente puede solicitar la devolucion o ca...,"Sí, el cliente puede solicitar la devolución o...","Sí, si presenta defecto de fábrica, dentro de ..."
1,¿Cuándo puede un huésped cancelar su reservaci...,"**En términos generales, un huésped puede canc...",Hasta 48 horas antes de la fecha de llegada.
2,¿Qué sucede si un paquete no puede ser entregado?,Si un paquete no puede entregarse en el primer...,Se harán hasta dos intentos de entrega.


## Conclusión y Análisis de Alucinaciones

El análisis de las tres preguntas demuestra claramente la superioridad del enfoque RAG (Retrieval Augmented Generation) para generar respuestas precisas y confiables en comparación con las respuestas directas del modelo (sin RAG).

### Análisis Detallado de Alucinaciones:

**Pregunta 1: "¿El cliente puede solicitar la devolucion o cambio del producto?"**
*   **Respuesta sin RAG:** El modelo proporcionó una política de devoluciones muy genérica y detallada (ej. plazos de 30 días, estado del producto en empaque original, múltiples motivos de devolución). Esto es una *alucinación* ya que no coincide con nuestra política específica.
*   **Respuesta con RAG:** "Sí, si presenta defecto de fábrica, dentro de 15 días y con ticket." Esta respuesta es precisa y concisa, basada directamente en nuestro documento.

**Pregunta 2: "¿Cuándo puede un huésped cancelar su reservación sin penalización?"**
*   **Respuesta sin RAG:** El modelo ofreció una explicación extensa sobre políticas de cancelación típicas (ej. "tarifa flexible", "no reembolsable", plazos de 24-48 horas, consejos). Esta es una *alucinación* al incorporar escenarios genéricos de la industria hotelera que no están en nuestra política específica.
*   **Respuesta con RAG:** "Hasta 48 horas antes de la fecha de llegada." Esta respuesta es directa y fiel a nuestro documento.

**Pregunta 3: "¿Qué sucede si un paquete no puede ser entregado?"**
*   **Respuesta sin RAG:** El modelo detalló un proceso general de entrega fallida, incluyendo múltiples intentos, retención en oficina de correos, devolución al remitente y situaciones especiales. Esta es una *alucinación* al proporcionar un flujo de trabajo de entrega genérico que va mucho más allá de nuestra política específica.
*   **Respuesta con RAG:** "Se harán hasta dos intentos de entrega." Esta respuesta se basa en nuestro documento.

### Conclusión General:

En todos los casos, las respuestas sin RAG presentan una cantidad significativa de "alucinaciones", generando información que parece plausible pero no está basada en los documentos específicos que le proporcionamos. En contraste, las respuestas con RAG son notablemente más concisas y precisas, limitándose estrictamente al contenido de los fragmentos recuperados. Esto demuestra la efectividad de RAG para anclar la respuesta dentro de un dominio de conocimiento definido, garantizando la exactitud y evitando la desinformación.

In [16]:
# Guardar el DataFrame de comparación en un archivo CSV
df_comparacion.to_csv('comparacion_rag_sin_rag.csv', index=False)
print("DataFrame 'df_comparacion' guardado en 'comparacion_rag_sin_rag.csv'")

DataFrame 'df_comparacion' guardado en 'comparacion_rag_sin_rag.csv'


**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

## Resumen Ejecutivo del Experimento: RAG vs. Generación sin RAG

### Objetivo del Experimento
El objetivo principal de este experimento fue comparar la calidad y la precisión de las respuestas generadas por un modelo de lenguaje con y sin la implementación de Retrieval Augmented Generation (RAG). Específicamente, buscamos identificar la presencia de "alucinaciones" en las respuestas sin RAG y demostrar cómo RAG ayuda a anclar las respuestas a una base de conocimiento específica, mejorando así la fidelidad.

### Metodología
1.  **Base de Conocimiento:** Se definió una pequeña base de conocimiento compuesta por tres fragmentos de texto que representaban políticas específicas (devoluciones de productos, cancelación de reservas, entrega de paquetes).
2.  **Embeddings:** Se utilizó la librería `sentence-transformers` con el modelo `paraphrase-multilingual-MiniLM-L12-v2` para generar embeddings vectoriales de estos documentos.
3.  **Función de Recuperación (`buscar_fragmento`):** Se implementó una función para calcular la similitud coseno entre una pregunta y los embeddings de los documentos, recuperando el fragmento más relevante.
4.  **Generación de Respuestas:**
    *   **Sin RAG:** Se enviaron tres preguntas al modelo de lenguaje directamente, sin proporcionar ningún contexto externo.
    *   **Con RAG:** Para las mismas tres preguntas, se recuperó primero el fragmento más relevante de la base de conocimiento y luego se envió este fragmento junto con la pregunta al modelo.
5.  **Comparación y Análisis:** Las respuestas generadas con ambos métodos se recopilaron y organizaron en un DataFrame de Pandas para facilitar la comparación. Posteriormente, se realizó un análisis cualitativo para identificar alucinaciones y evaluar la precisión de cada tipo de respuesta.

### Hallazgos Clave
*   **Respuestas sin RAG:** Demostraron una tendencia significativa a "alucinar". El modelo, al carecer de información específica, generó respuestas plausibles y detalladas, pero a menudo incorrectas o demasiado genéricas en relación con las políticas definidas. Estas alucinaciones incluían detalles inexistentes sobre plazos, procesos o condiciones que no se encontraban en la base de conocimiento.
*   **Respuestas con RAG:** Fueron consistentemente concisas, precisas y directamente fundamentadas en el contexto recuperado. La inclusión del fragmento relevante en el prompt del modelo eliminó efectivamente las alucinaciones, asegurando que las respuestas fueran fieles a las políticas.

### Conclusión
El experimento concluye que la implementación de RAG es fundamental para sistemas que requieren respuestas precisas y basadas en hechos de una base de conocimiento específica. RAG actúa como un mecanismo eficaz para mitigar las alucinaciones de los modelos de lenguaje, anclando sus respuestas a evidencia concreta. Esto es crucial para aplicaciones donde la exactitud y la confianza en la información son primordiales, como asistentes de políticas o chatbots de atención al cliente.